# NN-KNN on the Cartpole dataset

https://gymnasium.farama.org/introduction/basic_usage/

## Setup:

Basic imports and setups

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import sys
print(sys.executable)


/usr/bin/python3


Aliases to fix collions with hugging face

In [ ]:
import os, sys
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets as tv_datasets, transforms
from torch.utils.data import DataLoader, random_split

# Detect Colab
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    # Assumes drive is already mounted in a previous cell
    PROJECT_ROOT = Path("/content/drive/MyDrive/NN-kNN")
    DATA_ROOT    = Path("/content/datasets")
    CHECKPOINTS  = PROJECT_ROOT / "checkpoints"
    print("Running on Colab.")
else:
    PROJECT_ROOT = Path(os.getcwd())          # notebook's current folder as project root
    DATA_ROOT    = PROJECT_ROOT / "datasets"
    CHECKPOINTS  = PROJECT_ROOT / "checkpoints"
    print("Running locally.")

# Make project importable and set CWD to project root for consistent relative paths
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)

# --- FIX: avoid collision with HuggingFace `datasets` package ---
src = PROJECT_ROOT / "datasets"
dst = PROJECT_ROOT / "nnknn_datasets"

if src.exists() and (src / "reg_data.py").exists():
    # Only rename if the old folder exists and the new name doesn't
    if not dst.exists():
        os.rename(src, dst)
        print("Renamed:", src.name, "->", dst.name)
else:
    # If it's already renamed, that's fine
    pass

# Ensure package init exists (works for either case)
if dst.exists():
    (dst / "__init__.py").touch(exist_ok=True)
    print("Using dataset package folder:", dst)
else:
    # If you didn't rename (maybe you intentionally keep it), still ensure init
    if src.exists():
        (src / "__init__.py").touch(exist_ok=True)
        print("Using dataset package folder:", src)

# Import YOUR reg_data from the non-colliding name
from nnknn_datasets.reg_data import Reg_data, standardize_tensor

from model.nnknn_model import NN_KNN_Model, train_model, default_args, GlocalFeatureWeight

print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_ROOT     =", DATA_ROOT)
print("CHECKPOINTS  =", CHECKPOINTS)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Running on Colab.
Using dataset package folder: /content/drive/MyDrive/NN-kNN/nnknn_datasets


ModuleNotFoundError: No module named 'datasets.cls_small_data'

Cartpole imports

In [ ]:
import gymnasium as gym
import numpy as np

ENV_ID = "CartPole-v1"
env = gym.make(ENV_ID)

print("Gymnasium version:", gym.__version__)
print("Env:", ENV_ID)
print("Obs space:", env.observation_space)
print("Act space:", env.action_space)


### Config for the NN-Knn model

In [ ]:
cfg = {
    **default_args,
    "task_type": "regression",

    "case_score_mode": "bias_minus_distance",
    # "hard_knn",
    # "bias_minus_distance"
    # "neg_distance_logw"
    # "sigmoid"
    # "hard_knn"

    "softmax_over_cases": True,
    "tau": 1.0,
    "case_normalizer": "softmax",

    #for locality regularization in regression
    "regression_locality": False,
    "lambda_base": 1.0,
    "locality_alpha" : 2.0,
    "lambda_expdist": 0.1,

    "training_epochs": 200,#7000,
    "batch_size": 64,
    "feature_extractor_lr": 1e-3,
    "glocal_weightor_lr": 1e-3,
    "case_net_lr": 3e-4,
    "checkpoint_path": "nnknn_regression_best.pth",
    "patience": 80,

    "post_mlp_enabled": False,

    "nn_cdh_pretrain": True,
    "use_nn_cdh": True,
    "cdh_aggregate": True,

    "explanation_mode": True
}

# No feature extractor for this tabular demo
feature_extractor = None


## Dataset

This runs a policy (by default random), stores transitions, and assigns each (s,a) a Monte-Carlo discounted return.

In [ ]:
from dataclasses import dataclass

@dataclass
class CartpoleDatasetConfig:
    episodes: int = 300
    max_steps: int = 500
    gamma: float = 0.99
    seed: int = 0
    policy: str = "random"   # "random" or "heuristic"

def select_action(obs: np.ndarray, policy: str, rng: np.random.Generator) -> int:
    if policy == "random":
        return int(rng.integers(0, 2))
    elif policy == "heuristic":
        # simple classic: push in direction of pole angle
        # obs = [x, x_dot, theta, theta_dot]
        theta = obs[2]
        return 1 if theta > 0 else 0
    else:
        raise ValueError(f"Unknown policy={policy}")

def collect_cartpole_mc_dataset(env, ds_cfg: CartpoleDatasetConfig):
    rng = np.random.default_rng(ds_cfg.seed)

    X_list = []
    y_list = []

    for ep in range(ds_cfg.episodes):
        obs, info = env.reset(seed=int(rng.integers(0, 1_000_000)))
        traj = []  # (obs, action, reward)

        for t in range(ds_cfg.max_steps):
            a = select_action(obs, ds_cfg.policy, rng)
            next_obs, reward, terminated, truncated, info = env.step(a)

            traj.append((obs.copy(), int(a), float(reward)))
            obs = next_obs

            if terminated or truncated:
                break

        # Monte-Carlo returns G_t = r_t + gamma r_{t+1} + ...
        returns = []
        G = 0.0
        for (_, _, r) in reversed(traj):
            G = r + ds_cfg.gamma * G
            returns.append(G)
        returns.reverse()

        # Build supervised rows: features = [obs(4), action(1)], target = return
        for (step_i, ((s, a, r), Gt)) in enumerate(zip(traj, returns)):
            x = np.concatenate([np.asarray(s, dtype=np.float32), np.asarray([a], dtype=np.float32)], axis=0)
            X_list.append(x)
            y_list.append(np.float32(Gt))

    X = np.stack(X_list, axis=0)  # [N, 5]
    y = np.asarray(y_list, dtype=np.float32).reshape(-1, 1)  # [N, 1]
    return X, y

ds_cfg = CartpoleDatasetConfig(
    episodes=400,
    max_steps=500,
    gamma=0.99,
    seed=0,
    policy="heuristic",  # try "random" too
)

X_np, y_np = collect_cartpole_mc_dataset(env, ds_cfg)
print("Collected X:", X_np.shape, "y:", y_np.shape)
print("X sample:", X_np[0], "y sample:", y_np[0])


### Normailzation

In [ ]:
y_mean = y_train.mean(dim=0, keepdim=True)
y_std  = y_train.std(dim=0, keepdim=True)

y_train_norm = (y_train - y_mean) / (y_std + 1e-8)
y_val_norm   = (y_val   - y_mean) / (y_std + 1e-8)

print("y_mean:", y_mean.flatten()[:5], "y_std:", y_std.flatten()[:5])


y_mean: tensor([19.4430]) y_std: tensor([10.7677])


## Train

In [ ]:
best_acc, glocal_weightor, model = train_model(
    X_train, y_train_norm, X_val, y_val_norm,
    feature_extractor=feature_extractor,
    cfg=cfg
)

print('Best (R^2 proxy in logs).')


[NN-CDH-AGG] Epoch    1 train_loss=1.565600 val_loss=1.115112
[NN-CDH-AGG] Epoch    2 train_loss=1.015970 val_loss=0.991849
[NN-CDH-AGG] Epoch    3 train_loss=0.962688 val_loss=0.965268
[NN-CDH-AGG] Epoch    4 train_loss=0.945384 val_loss=0.952265
[NN-CDH-AGG] Epoch    5 train_loss=0.934958 val_loss=0.943731
[NN-CDH-AGG] Epoch    6 train_loss=0.926704 val_loss=0.936486
[NN-CDH-AGG] Epoch    7 train_loss=0.918633 val_loss=0.928895
[NN-CDH-AGG] Epoch    8 train_loss=0.910352 val_loss=0.920436
[NN-CDH-AGG] Epoch    9 train_loss=0.901790 val_loss=0.911280
[NN-CDH-AGG] Epoch   10 train_loss=0.892089 val_loss=0.901561
[NN-CDH-AGG] Epoch   11 train_loss=0.881226 val_loss=0.889941
[NN-CDH-AGG] Epoch   12 train_loss=0.869688 val_loss=0.879640
[NN-CDH-AGG] Epoch   13 train_loss=0.858694 val_loss=0.869608
[NN-CDH-AGG] Epoch   14 train_loss=0.849105 val_loss=0.861415
[NN-CDH-AGG] Epoch   15 train_loss=0.840187 val_loss=0.853173
[NN-CDH-AGG] Epoch   16 train_loss=0.832188 val_loss=0.845422
[NN-CDH-

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([64, 1])) that is different to the input size (torch.Size([64])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([29, 1])) that is different to the input size (torch.Size([29])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


[Stage1] Epoch 1 - Val R²: -3440.5192 | Reg Val Loss: 1.0245
[Stage1] New best (epoch 1) Val Loss: 1.0245 — saved: nnknn_regression_best_retr.pth
[Stage1] Epoch 2 - Val R²: -3408.4602 | Reg Val Loss: 1.0153
[Stage1] New best (epoch 2) Val Loss: 1.0153 — saved: nnknn_regression_best_retr.pth
[Stage1] Epoch 3 - Val R²: -3386.5310 | Reg Val Loss: 1.0092
[Stage1] New best (epoch 3) Val Loss: 1.0092 — saved: nnknn_regression_best_retr.pth
[Stage1] Epoch 4 - Val R²: -3371.7392 | Reg Val Loss: 1.0052
[Stage1] New best (epoch 4) Val Loss: 1.0052 — saved: nnknn_regression_best_retr.pth
[Stage1] Epoch 5 - Val R²: -3362.5983 | Reg Val Loss: 1.0027
[Stage1] New best (epoch 5) Val Loss: 1.0027 — saved: nnknn_regression_best_retr.pth
[Stage1] Epoch 6 - Val R²: -3359.1836 | Reg Val Loss: 1.0019
[Stage1] New best (epoch 6) Val Loss: 1.0019 — saved: nnknn_regression_best_retr.pth
[Stage1] Epoch 7 - Val R²: -3357.6841 | Reg Val Loss: 1.0016
[Stage1] New best (epoch 7) Val Loss: 1.0016 — saved: nnknn_reg

KeyboardInterrupt: 

## Sanity Check

In [ ]:
#Un normalize predictions and show a few rows
model.eval()

with torch.no_grad():
    # take a small batch from val
    xb = X_val[:32]
    yb_norm = y_val_norm[:32]

    # model output shape depends on your implementation; common is [B,1]
    pred_norm = model(xb) if feature_extractor is None else model(feature_extractor(xb))

    # Unnormalize back to return scale
    pred = pred_norm * (y_std + 1e-8) + y_mean
    y_true = yb_norm * (y_std + 1e-8) + y_mean

print("pred (first 10):", pred[:10].squeeze().cpu().numpy())
print("true (first 10):", y_true[:10].squeeze().cpu().numpy())


NameError: name 'model' is not defined

## Rendering and Comparisons

### Render

In [ ]:
# Make env renderable
import gymnasium as gym

ENV_ID = "CartPole-v1"

# IMPORTANT: for notebook display, use rgb_array
play_env = gym.make(ENV_ID, render_mode="rgb_array")


In [ ]:
import numpy as np
import torch

def predict_return_regressor(model, state_np: np.ndarray, action: int, device: torch.device) -> float:
    """
    model: takes x of shape [B, 5] where x = [obs(4), action(1)]
    returns: scalar predicted normalized return (if you trained on normalized y)
    """
    x = np.concatenate([state_np.astype(np.float32), np.array([action], dtype=np.float32)], axis=0)
    xt = torch.tensor(x, dtype=torch.float32, device=device).unsqueeze(0)  # [1,5]
    with torch.no_grad():
        yhat = model(xt)  # expected shape [1,1] or [1]
    return float(yhat.squeeze().detach().cpu().item())

def choose_action_greedy(model, state_np: np.ndarray, device: torch.device) -> int:
    """
    Greedy policy: choose action with higher predicted (normalized) return.
    """
    q0 = predict_return_regressor(model, state_np, 0, device)
    q1 = predict_return_regressor(model, state_np, 1, device)
    return int(1 if q1 > q0 else 0)


In [ ]:
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

def rollout_with_frames(env, policy_fn, max_steps=500, seed=0):
    obs, info = env.reset(seed=seed)
    frames = []
    total_reward = 0.0

    for t in range(max_steps):
        frame = env.render()  # rgb_array
        frames.append(frame)

        action = policy_fn(obs)
        obs, reward, terminated, truncated, info = env.step(action)
        total_reward += float(reward)

        if terminated or truncated:
            break

    return frames, total_reward

def show_frames_as_html(frames, interval_ms=30):
    fig = plt.figure()
    im = plt.imshow(frames[0])
    plt.axis("off")

    def animate(i):
        im.set_data(frames[i])
        return [im]

    anim = animation.FuncAnimation(
        fig, animate, frames=len(frames), interval=interval_ms, blit=True
    )
    plt.close(fig)
    return HTML(anim.to_jshtml())


### Compare

In [ ]:
# Basic MLP
import torch.nn as nn
import torch.optim as optim

class MLPRegressor(nn.Module):
    def __init__(self, in_dim=5, hidden=(64, 64)):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden:
            layers.append(nn.Linear(d, h))
            layers.append(nn.ReLU())
            d = h
        layers.append(nn.Linear(d, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)

baseline = MLPRegressor(in_dim=5, hidden=(64, 64)).to(device)

opt = optim.Adam(baseline.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

def train_baseline(baseline, X_train, y_train_norm, X_val, y_val_norm, epochs=50, batch_size=256):
    baseline.train()
    N = X_train.shape[0]
    best_val = float("inf")
    best_state = None

    for ep in range(1, epochs + 1):
        perm = torch.randperm(N, device=X_train.device)
        Xs = X_train[perm]
        ys = y_train_norm[perm]

        for i in range(0, N, batch_size):
            xb = Xs[i:i+batch_size].to(device)
            yb = ys[i:i+batch_size].to(device)

            opt.zero_grad(set_to_none=True)
            pred = baseline(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()

        baseline.eval()
        with torch.no_grad():
            val_pred = baseline(X_val.to(device))
            val_loss = loss_fn(val_pred, y_val_norm.to(device)).item()
        baseline.train()

        if val_loss < best_val:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in baseline.state_dict().items()}

        if ep % 10 == 0 or ep == 1:
            print(f"Baseline epoch {ep:3d} | val_mse={val_loss:.6f} | best={best_val:.6f}")

    if best_state is not None:
        baseline.load_state_dict(best_state)
    baseline.eval()
    return baseline

baseline = train_baseline(baseline, X_train, y_train_norm, X_val, y_val_norm, epochs=80, batch_size=512)


Baseline epoch   1 | val_mse=0.814787 | best=0.814787
Baseline epoch  10 | val_mse=0.295467 | best=0.295467
Baseline epoch  20 | val_mse=0.244842 | best=0.244842
Baseline epoch  30 | val_mse=0.229805 | best=0.229805
Baseline epoch  40 | val_mse=0.228566 | best=0.227479
Baseline epoch  50 | val_mse=0.225187 | best=0.224789
Baseline epoch  60 | val_mse=0.223201 | best=0.223201
Baseline epoch  70 | val_mse=0.223558 | best=0.221841
Baseline epoch  80 | val_mse=0.221662 | best=0.221662


In [ ]:
import numpy as np

def evaluate_policy(env_id, policy_fn, episodes=20, seed=0, max_steps=500):
    eval_env = gym.make(env_id)  # no render for fast eval
    returns = []
    rng = np.random.default_rng(seed)

    for ep in range(episodes):
        obs, info = eval_env.reset(seed=int(rng.integers(0, 1_000_000)))
        total = 0.0
        for t in range(max_steps):
            a = policy_fn(obs)
            obs, r, terminated, truncated, info = eval_env.step(a)
            total += float(r)
            if terminated or truncated:
                break
        returns.append(total)

    eval_env.close()
    return np.array(returns, dtype=np.float32)

# policy wrappers
nnknn_policy = lambda obs: choose_action_greedy(model, obs, device)
base_policy  = lambda obs: choose_action_greedy(baseline, obs, device)
rand_policy  = lambda obs: int(np.random.randint(0, 2))

nnknn_returns = evaluate_policy(ENV_ID, nnknn_policy, episodes=30, seed=1)
base_returns  = evaluate_policy(ENV_ID, base_policy,  episodes=30, seed=1)
rand_returns  = evaluate_policy(ENV_ID, rand_policy,  episodes=30, seed=1)

print("NN-KNN  mean:", nnknn_returns.mean(), "std:", nnknn_returns.std(), "min/max:", nnknn_returns.min(), nnknn_returns.max())
print("Baseline mean:", base_returns.mean(),  "std:", base_returns.std(),  "min/max:", base_returns.min(),  base_returns.max())
print("Random  mean:", rand_returns.mean(),  "std:", rand_returns.std(),  "min/max:", rand_returns.min(),  rand_returns.max())


NameError: name 'model' is not defined

In [ ]:
# NN-KNN play
frames, total = rollout_with_frames(
    play_env,
    policy_fn=lambda obs: choose_action_greedy(model, obs, device),
    seed=42
)
print("NN-KNN episode return:", total)
show_frames_as_html(frames, interval_ms=30)


In [ ]:
frames, total = rollout_with_frames(
    play_env,
    policy_fn=lambda obs: choose_action_greedy(baseline, obs, device),
    seed=42
)
print("Baseline episode return:", total)
show_frames_as_html(frames, interval_ms=30)
